# 0. 2-factor productivity

In [ ]:
out_file_name = "results_0_tfp"

table_panel_name = "working_yearly"
table_panel_old = "fame_yearly_kp"

t_panel = con.table(table_panel_name)
t_old = con.table(table_panel_old)
df_panel = (
    t_panel
    .drop('gva1_per_worker', 'gva2_per_worker')
    .distinct(on=['registered_number', 'year'])
    .left_join(
        t_old.select('registered_number', 'year', 'tangibles', 'intangibles', 'investments_other').distinct(on=['registered_number', 'year']),
        ['registered_number', 'year']
    )
    .execute()
)

models = {
    'gva1_ft': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'gva1_f': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i']},
    'gva1_t': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['t']},
    'gva1_n': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': []}
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Running model 'gva1_ft'
✅ Model 'gva1_ft' estimated: ln_total_assets=0.286, ln_employees=0.617
Running model 'gva1_f'
✅ Model 'gva1_f' estimated: ln_total_assets=0.284, ln_employees=0.613
Running model 'gva1_t'
✅ Model 'gva1_t' estimated: ln_total_assets=0.421, ln_employees=0.562
Running model 'gva1_n'
✅ Model 'gva1_n' estimated: ln_total_assets=0.421, ln_employees=0.561
Panel regressions complete. 4 models, writing.
